# 드럼 채보 벤치마크 — 코랩 실행 (S15P31A307-110)

저장소 `AI/` 코드와 구글 드라이브의 데이터로 `ssaplash-bench run-transcription` 명령을 그대로 돌린다 (채보 설계 4-6절·#10 계획). 셀마다 명령만 부르고 로직은 없다. 결과·MLflow 기록은 드라이브에 바로 써서 세션이 끊겨도 남는다. 분리 벤치마크 노트북(`bench_colab.ipynb`)과는 별개다.

준비물 (드라이브 `MyDrive/ssaplash/`)
- `S15P31A307.zip`: 저장소를 zip으로 묶은 것 (`git archive`, README '코랩에서 돌리기')
- `data/synth_v1/`: 합성곡 5곡 (216MB). 분리 벤치마크 때 올린 것 그대로
- `data/gmd/groove/`: GMD test 연주 45개 + info.csv (약 560MB). `scripts/pack_gmd_subset.py`로 뽑은 `data/upload/gmd/` 폴더를 `data/gmd/`로 올린다
- `data/mdb_drums/`: 로컬 `AI/data/mdb_drums/` 폴더 그대로 (`MDB Drums/` 23곡, 331MB)

런타임은 GPU(코랩 프로+ A100 또는 L4)로 바꾼 뒤 위에서부터 실행한다. GPU는 한 종류로만 측정한다. 4번 셀(별도 환경 3개)은 세션마다 다시 만들어야 하고 10~20분이 든다.

**세션이 끊기면** 런타임을 다시 연결하고 1~6을 다시 실행한 뒤 끊긴 본측정 셀을 그대로 다시 실행한다. runId가 고정돼 있어 이미 끝난 곡은 건너뛰고 실패한 곡만 다시 돈다(이어하기). 3번 셀이 찍는 GPU 이름이 이전과 다르면 결과 CSV의 `gpuName` 열이 섞이므로, 런타임을 바꿔 같은 GPU를 받은 뒤 이어 간다.


In [ ]:
# 1. 드라이브 연결
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ssaplash"
!ls -la $DRIVE $DRIVE/data

In [ ]:
# 2. 저장소 풀기 (zip 방식). 이미 풀려 있으면 건너뜀. 코드를 새로 올렸으면 /content/S15P31A307 을 지우고 다시 실행
import os
if not os.path.isdir("/content/S15P31A307/AI"):
    !unzip -q "$DRIVE/S15P31A307.zip" -d /content/
%cd /content/S15P31A307/AI
!git log --oneline -1 2>/dev/null || echo "git 정보 없음 (zip)"

In [ ]:
# 3. uv 설치 + 의존성 (분리 모델 extra + 채보 모델 ADTOF extra). 한글 그래프용 글꼴도 설치. 마지막 줄의 GPU 이름을 기록해 둔다
# 코랩 커널의 MPLBACKEND(matplotlib_inline)는 uv 가상환경에 없는 모듈이라 Agg로 덮어씀 (cli도 같은 처리)
%env MPLBACKEND=Agg
!pip install -q uv
!apt-get install -qq -y fonts-nanum > /dev/null
!uv sync --extra models --extra transcribers
!uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
# 4. 별도 환경 3개 (ADT_STR · Inverse Drum Machine · Omnizart). README '별도 환경 만들기'의 Linux 판
# 이미 만들어져 있으면 건너뜀. yaml의 python_exe는 Windows 경로지만 하네스가 .venv-<이름>/bin/python을 자동으로 찾는다
# ADT_STR: 저장소 클론 + torch 2.8 + 저장소 의존성. 가중치(HF Pierfrancesco/adt-str, 1.1GB)는 첫 실행 때 받는다
![ -d ext/ADT_STR ] || git clone --depth 1 https://github.com/pier-maker92/ADT_STR ext/ADT_STR
![ -f .venv-adt-str/bin/python ] || (uv venv .venv-adt-str --python 3.11 \
    && uv pip install --python .venv-adt-str/bin/python torch==2.8.0 torchaudio==2.8.0 \
    && uv pip install --python .venv-adt-str/bin/python -r ext/ADT_STR/pyproject.toml)
# Inverse Drum Machine: 저장소 클론(가중치 포함) + 의존성만 설치 (패키지로는 설치가 안 됨)
![ -d ext/inverse-drum-machine ] || git clone --depth 1 https://github.com/bernardo-torres/inverse-drum-machine ext/inverse-drum-machine
![ -f .venv-idm/bin/python ] || (uv venv .venv-idm --python 3.11 \
    && uv pip install --python .venv-idm/bin/python "torch>=2.1.1,<2.9" "torchaudio>=2.1.1,<2.9" \
    && uv pip install --python .venv-idm/bin/python lightning lightning-utilities auraloss einops "librosa>=0.9.1,<0.10" soundfile "soxr>=0.3.7" mir_eval "numpy<2" scipy scikit-learn pandas "matplotlib==3.8.2" hydra-core hydra-colorlog omegaconf pyyaml rootutils tqdm rich beautifulsoup4 "setuptools<81")
# Omnizart: Python 3.10 + TensorFlow. madmom이 Cython·옛 numpy로 빌드돼야 해서 먼저 깔아 둔다
![ -f .venv-omnizart/bin/python ] || (uv python install 3.10 && uv venv .venv-omnizart --python 3.10 \
    && uv pip install --python .venv-omnizart/bin/python "Cython<3" "numpy==1.23.5" wheel setuptools \
    && uv pip install --python .venv-omnizart/bin/python --no-build-isolation "madmom==0.16.1" \
    && uv pip install --python .venv-omnizart/bin/python omnizart soundfile \
    && .venv-omnizart/bin/omnizart download-checkpoints)
!ls -d .venv-*/bin/python ext/*

In [ ]:
# 5. 데이터를 런타임 로컬 디스크로 복사 (드라이브 직접 읽기는 느리고 긴 실행 중 끊김). 이미 있으면 건너뜀
!mkdir -p data
![ -d data/synth_v1 ] || cp -r "$DRIVE/data/synth_v1" data/
![ -d data/gmd ] || cp -r "$DRIVE/data/gmd" data/
![ -d data/mdb_drums ] || cp -r "$DRIVE/data/mdb_drums" data/
!du -sh data/synth_v1 data/gmd data/mdb_drums
!PYTHONUTF8=1 uv run ssaplash-bench check-data configs/bench/datasets/synth_v1.yaml configs/bench/datasets/gmd_test.yaml configs/bench/datasets/mdb_drums.yaml configs/bench/datasets/mdb_drums_unseen.yaml

In [ ]:
# 6. 본측정에 넣는 채보 모델 목록. 조건부 모델(YourMT3+·Vogl 2018·OaF)이 붙으면 여기에 yaml을 더한다
ALL = " ".join(f"configs/bench/transcribers/{m}.yaml" for m in [
    "adtof/adtof", "adtof_drumsep/adtof_drumsep",
    "external/adt_str_tau08", "external/idm", "external/omnizart",
])

In [ ]:
# 7. 동작 확인: 모델 전부 × 합성곡 5곡 30초 클립. 별도 환경 3개가 전부 error 없이 행을 남기는지, 결과·MLflow가 드라이브에 써지는지 본다
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models $ALL --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" \
    --dataset configs/bench/datasets/synth_v1.yaml --clip 0 30 --run-id clip30-transcription
!cat "$DRIVE/results/clip30-transcription/summary.md"

## 본측정 (정답 드럼 트랙 입력)

runId가 고정돼 있어 끊기면 같은 셀을 다시 실행하면 이어서 돈다. `--save-midi N`은 모델당 N곡의 예측 타격을 MIDI로 저장해 나중에 플레이해 볼 수 있게 한다. gmd_test(음원 52분)가 가장 긴 run이다.

In [ ]:
# 8. 합성곡 5곡
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models $ALL --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" \
    --dataset configs/bench/datasets/synth_v1.yaml --run-id drums-synth --save-midi 5

In [ ]:
# 9. GMD test 연주 45개 (8패드 점수의 주 데이터)
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models $ALL --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" \
    --dataset configs/bench/datasets/gmd_test.yaml --run-id drums-gmd --save-midi 3

In [ ]:
# 10. MDB Drums 23곡 (논문 수치와 검산)
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models $ALL --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" \
    --dataset configs/bench/datasets/mdb_drums.yaml --run-id drums-mdb --save-midi 3

In [ ]:
# 11. MDB Drums 중 분리 모델이 학습 때 못 본 11곡. 아래 원곡·분리 드럼 입력과 비교하는 기준 행
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models $ALL --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" \
    --dataset configs/bench/datasets/mdb_drums_unseen.yaml --run-id drums-mdbu

## 본측정 (원곡 입력, 분리 모델이 뽑은 드럼 입력)

같은 모델에 원곡을 그대로 넣었을 때와 분리 모델(S15P31A307-105 상위 2개)이 뽑은 드럼을 넣었을 때 점수가 얼마나 떨어지는지 본다. 분리한 드럼은 run 폴더 `separated/`에 저장돼 모델을 더해도 다시 분리하지 않는다. GMD는 반주가 없어 여기서는 빼고 합성곡·MDB 11곡만 쓴다.

In [ ]:
# 12. 원곡 입력
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models $ALL --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" \
    --dataset configs/bench/datasets/synth_v1.yaml --input mixture --run-id mix-synth
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models $ALL --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" \
    --dataset configs/bench/datasets/mdb_drums_unseen.yaml --input mixture --run-id mix-mdbu

In [ ]:
# 13. BS Roformer SW가 뽑은 드럼 입력 (가중치는 첫 실행 때 data/models/ 에 받는다)
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models $ALL --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" \
    --dataset configs/bench/datasets/synth_v1.yaml --input separated --separator configs/bench/models/msst/bs_roformer_sw.yaml --run-id sep-sw-synth
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models $ALL --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" \
    --dataset configs/bench/datasets/mdb_drums_unseen.yaml --input separated --separator configs/bench/models/msst/bs_roformer_sw.yaml --run-id sep-sw-mdbu

In [ ]:
# 14. BS Roformer 공개 데이터판이 뽑은 드럼 입력
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models $ALL --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" \
    --dataset configs/bench/datasets/synth_v1.yaml --input separated --separator configs/bench/models/msst/bs_roformer_musdb.yaml --run-id sep-musdb-synth
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models $ALL --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" \
    --dataset configs/bench/datasets/mdb_drums_unseen.yaml --input separated --separator configs/bench/models/msst/bs_roformer_musdb.yaml --run-id sep-musdb-mdbu

In [ ]:
# 15. 보조: ADT_STR의 문턱값 변형 tau 0.4·0.6 (비교표 본문에는 저장소 기본값 tau 0.8만 넣는다)
!PYTHONUTF8=1 uv run ssaplash-bench run-transcription --models configs/bench/transcribers/external/adt_str_tau04.yaml configs/bench/transcribers/external/adt_str_tau06.yaml \
    --dataset configs/bench/datasets/gmd_test.yaml --out "$DRIVE/results" --mlflow-uri "$DRIVE/mlruns" --run-id drums-gmd-tau

In [ ]:
# 16. 본측정 run 전부를 합친 보고서 → 드라이브 results/combined/ (tau 변형은 results/combined-tau/)
RUNS = " ".join(f"{DRIVE}/results/{r}/results.csv" for r in [
    "drums-synth", "drums-gmd", "drums-mdb", "drums-mdbu", "mix-synth", "mix-mdbu",
    "sep-sw-synth", "sep-sw-mdbu", "sep-musdb-synth", "sep-musdb-mdbu",
])
!PYTHONUTF8=1 uv run ssaplash-bench report-transcription $RUNS --out "$DRIVE/results/combined"
!PYTHONUTF8=1 uv run ssaplash-bench report-transcription "$DRIVE/results/drums-gmd-tau/results.csv" "$DRIVE/results/drums-gmd/results.csv" --out "$DRIVE/results/combined-tau"
!cat "$DRIVE/results/combined/summary.md"

로컬에서 보고서를 다시 만들려면 드라이브의 `results/<runId>/` 폴더(results.csv·hits_by_class.csv)를 받아 `uv run ssaplash-bench report-transcription <csv ...>`를 돌린다. MLflow 기록은 `mlruns/` 폴더째 받아 `uv run mlflow ui --backend-store-uri mlruns`로 본다. 결과 CSV의 `gpuName` 열이 한 종류인지, `error` 열이 비어 있는지 확인한 뒤 문서에 쓴다.